# Client Credentials (service) login → a **Delta** table

Authenticate with the **OAuth2 `client_credentials` grant** — machine-to-machine, no human in
the loop, ideal for jobs and services — then create a Delta generic table and round-trip data
through `deltalake` with vended S3 credentials.

> **Kernel:** run this with the **Python (pylakekeeper)** kernel (the `python/.venv`). If you
> see `ModuleNotFoundError: No module named 'pylakekeeper'`, you're on the wrong kernel — see
> `examples/README.md` → *Running the notebooks*.

## Requirements
```sh
cd python && pip install -e '.[examples]' deltalake
```
A Lakekeeper with an OAuth2 IdP and an existing warehouse + namespace. Unlike the interactive
notebooks this uses a **confidential** client id + secret (e.g. the Keycloak `spark` client).

In [ ]:
import os

KEYCLOAK = os.environ.get("KEYCLOAK", "http://localhost:30080")
REALM = os.environ.get("KEYCLOAK_REALM", "iceberg")
TOKEN_URL = os.environ.get(
    "OAUTH_TOKEN_URL", f"{KEYCLOAK}/realms/{REALM}/protocol/openid-connect/token"
)
CLIENT_ID = os.environ.get("OAUTH_CLIENT_ID", "spark")
CLIENT_SECRET = os.environ.get("OAUTH_CLIENT_SECRET", "")  # set it in the environment
# REQUIRED: service-account clients (spark/trino) don't carry the `lakekeeper` audience by
# default, so without this scope the token has aud=account and Lakekeeper rejects it (401).
SCOPE = os.environ.get("OAUTH_SCOPE", "lakekeeper")

# --- Lakekeeper ---
LAKEKEEPER = os.environ.get("LAKEKEEPER", "http://localhost:8181")
WAREHOUSE_ID = os.environ.get("WAREHOUSE_ID", "")  # the warehouse UUID (URL prefix, not name)
PROJECT_ID = os.environ.get("PROJECT_ID") or None
NAMESPACE = os.environ.get("NAMESPACE", "ai.test")

In [ ]:
import base64
import json
import time


def token_seconds_left(auth_header: str) -> int:
    """Decode a Bearer JWT's `exp` claim and return seconds until it expires."""
    token = auth_header.split(" ", 1)[-1]
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)  # restore base64 padding
    claims = json.loads(base64.urlsafe_b64decode(payload))
    return int(claims["exp"] - time.time())

## Authenticate (client credentials)
No browser — the token is fetched (and later refreshed) programmatically.

In [ ]:
from pylakekeeper import ClientCredentials

auth = ClientCredentials(
    token_url=TOKEN_URL,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope=SCOPE,
)
print("Authenticated ✅  token expires in", token_seconds_left(auth.auth_header()), "s")

## Sample data

In [ ]:
import numpy as np
import pyarrow as pa

EMBED_DIM = 8
ROWS = 100
_rng = np.random.default_rng(42)
_embeddings = _rng.standard_normal((ROWS, EMBED_DIM)).astype(np.float32)

sample = pa.table(
    {
        "id": pa.array(range(ROWS), type=pa.int64()),
        "sku": pa.array([f"SKU-{i:06d}" for i in range(ROWS)]),
        "embedding": pa.FixedSizeListArray.from_arrays(
            pa.array(_embeddings.reshape(-1), type=pa.float32()), EMBED_DIM
        ),
    }
)
print(sample.schema)

## Create the Delta table and write it

`deltalake` uses `object_store`-style `storage_options`. We remap the vended Iceberg
credentials to Delta's key names; `AWS_S3_ALLOW_UNSAFE_RENAME=true` lets delta-rs commit to
S3 without an external locking provider (fine for a single-writer example).

In [ ]:
from deltalake import write_deltalake

from pylakekeeper import Client, ConflictError, GenericTableFormat

TABLE = os.environ.get("TABLE", "delta_embeddings")


def delta_storage_options(resp):
    lo = resp.lance_storage_options  # already object_store-shaped (aws_* keys)
    opts = {
        "AWS_ACCESS_KEY_ID": lo.get("aws_access_key_id"),
        "AWS_SECRET_ACCESS_KEY": lo.get("aws_secret_access_key"),
        "AWS_SESSION_TOKEN": lo.get("aws_session_token"),
        "AWS_REGION": lo.get("aws_region", "local-01"),
        "AWS_ENDPOINT_URL": lo.get("aws_endpoint"),
        "AWS_ALLOW_HTTP": "true" if lo.get("allow_http") else "false",
        "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    }
    return {k: v for k, v in opts.items() if v is not None}


with Client(base_url=LAKEKEEPER, warehouse=WAREHOUSE_ID, auth=auth, project_id=PROJECT_ID) as c:
    try:
        c.generic_tables.create(
            NAMESPACE,
            TABLE,
            format=GenericTableFormat.DELTA,
            properties={"embedding-dim": str(EMBED_DIM)},
        )
    except ConflictError:
        print(f"{NAMESPACE}.{TABLE} already exists — continuing")

    resp = c.generic_tables.load(NAMESPACE, TABLE, vended=True)
    print("location =", resp.location)
    write_deltalake(
        resp.location, sample, storage_options=delta_storage_options(resp), mode="overwrite"
    )

## Read the Delta table back

Re-load for **fresh** vended credentials, then open with `deltalake` using the same
`delta_storage_options` mapping. Delta is versioned — `dt.version()` shows the snapshot.

In [ ]:
from deltalake import DeltaTable

with Client(base_url=LAKEKEEPER, warehouse=WAREHOUSE_ID, auth=auth, project_id=PROJECT_ID) as c:
    resp = c.generic_tables.load(NAMESPACE, TABLE, vended=True)  # fresh short-lived creds

dt = DeltaTable(resp.location, storage_options=delta_storage_options(resp))
print("version   =", dt.version())
back = dt.to_pyarrow_table()
print("row_count =", back.num_rows)
print(back.select(["id", "sku"]).slice(0, 5))

## The session outlives the access token
`ClientCredentials` re-fetches automatically before expiry — no manual refresh.

In [ ]:
before = token_seconds_left(auth.auth_header())
print("current access token expires in", before, "s")

auth.invalidate()  # what the transport does on a 401 — simulate expiry

after = token_seconds_left(auth.auth_header())  # no re-login: silent refresh
print("after refresh, new token expires in", after, "s")
assert after >= before, "expected a freshly-minted token"
print("\n✅ Session renewed via refresh_token — no re-authentication.")